# FiftyOne workflow: your computer (viewer and labels)

Browse the tiles and the labels, tag test tiles, compare the models, look at the embeddings, and add new labels. This notebook needs **no torch**: it only uses the `fo` group of `pyproject.toml`.

Start it from the repository with `uv run --group fo jupyter lab`, then open this notebook.

**Data used (explicit):** `data/tiles_10cm/` and `data/tiles_10cm/labels/`, the same as `model_train.ipynb`.

## 1. Setup

The same start as `model_train.ipynb`.

In [ ]:
# Same start in both notebooks. On Colab or Kaggle this downloads the repository. On your computer
# the repository is already here and nothing is downloaded.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/emanuel-gf/FTL-Model-Training.git"   # set to your repository
REPO_DIR = "FTL-Model-Training"
IN_CLOUD = "COLAB_RELEASE_TAG" in os.environ or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
os.environ["MPLBACKEND"] = "Agg"

if not Path("pyproject.toml").exists():
    if not Path(REPO_DIR).exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif IN_CLOUD and Path(".git").exists():
    subprocess.run(["git", "pull", "--ff-only"], check=False)
print("repository folder:", Path.cwd())

In [ ]:
# Same in both notebooks. Only GROUP differs: "gpu" in the cloud, "fo" on your computer.
from src.bootstrap import setup

GROUP = "fo"
TILES_DIR = "data/tiles_10cm"       # EXPLICIT: the 10 cm tiles and their GeoJSON labels, used by both notebooks
TILES = setup(GROUP, TILES_DIR)

## 2. Tiles and labels (uv)

The same folder as the cloud notebook. The tiling runs again only if the tiles are missing. The labels are always rebuilt from `data/labelme/images/`.

In [ ]:
RETILE = not (TILES / "tiles_index.csv").exists()          # True: tile the COGs now
if RETILE:
    if IN_CLOUD:
        !uv run python download_images.py --list list_of_ids.txt --out cog
    !uv run python tile_cogs.py --input cog --gsd 0.10 --tile 640 --overlap 128 --output {TILES_DIR}

!uv run python export_annotations.py --index {TILES_DIR}/tiles_index.csv --labels data/labelme/images --out {TILES_DIR}/labels --tile 640

## 3. FiftyOne dataset

One sample per tile. The labels are the field `ground_truth`.

In [ ]:
from src.fo_dataset import (build_dataset, dataset_summary, tag_test_samples, add_predictions,
                           evaluate_run, compare_runs, attach_embeddings)

dataset = build_dataset(tiles_dir=TILES_DIR, labels_dir=f"{TILES_DIR}/labels", overwrite=True)
dataset_summary(dataset)

## 4. Test tiles

A few annotated tiles tagged `test`. The model predictions are compared on these tiles only.

In [ ]:
test_view = tag_test_samples(dataset, n=10, seed=51)

## 5. Predictions from the cloud

Reads `outputs/predictions/<name>/` for the test tiles. First unzip `results_for_local.zip` (from the cloud notebook, step 8) at the root of the repository.

In [ ]:
import zipfile
from pathlib import Path

if Path("results_for_local.zip").exists() and not Path("outputs/predictions").exists():
    zipfile.ZipFile("results_for_local.zip").extractall(".")

summary = {}
for folder in sorted(Path("outputs/predictions").glob("*")):
    add_predictions(test_view, folder.name, folder)
    summary[folder.name] = evaluate_run(test_view, folder.name)
results_table = compare_runs(summary)

## 6. Embeddings from the cloud

Links `data/embeddings/10cm/dinov2_base.npz` to the samples by `tile_id`, after checking that the resolution, the tile size and the bounds match. Then it computes a **t-SNE** map in FiftyOne Brain (the method of notebook 08), stored under `dinov2_tsne`.

In [ ]:
attach_embeddings(dataset, "data/embeddings/10cm/dinov2_base.npz")


## 7. Clustering and uniqueness (from notebook 09)

Same steps as `09_clustering_uniqueness.ipynb` in webinar-DL, with the same functions (`src/active_learning.py`, copied unchanged). The vectors come from the embedding file and are passed as arrays, so they stay out of MongoDB.

- **Clustering**: k-means on the embeddings, saved as the field `cluster_kmeans_<N>` (strings, so FiftyOne treats them as categories).
- **Global uniqueness**: how isolated a tile is among its *k* nearest neighbours in the whole dataset (field `uniqueness_score`).
- **Uniqueness inside each cluster**: the same score, only among the tiles of the same cluster (field `uniqueness_in_cluster_<N>`).

Choose `N_CLUSTERS` and `K_NEIGHBOURS` for the number of tiles: notebook 09 used 12 clusters for ~1,755 tiles and k = 10 (k must stay below √N).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.fo_dataset import aligned_embeddings
from src.active_learning import compute_clustering, compute_uniqueness_field, compute_uniqueness_field_v2

EMB_FILE = "data/embeddings/10cm/dinov2_base.npz"
BRAIN_KEY = "dinov2_tsne"
N_CLUSTERS = 12
K_NEIGHBOURS = 10
CLUSTER_FIELD = f"cluster_kmeans_{N_CLUSTERS}"
UNIQUENESS_FIELD = "uniqueness_score"
UNIQUENESS_CLUSTER_FIELD = f"uniqueness_in_cluster_{N_CLUSTERS}"

emb, sample_ids, _ = aligned_embeddings(dataset, EMB_FILE)
print(f"embeddings aligned to the dataset: {emb.shape}, k = {K_NEIGHBOURS} < sqrt(N) = {len(sample_ids) ** 0.5:.1f}")

In [ ]:
cluster_labels, _, clustered_ids = compute_clustering(
    dataset,
    embeddings=emb,
    sample_ids=sample_ids,
    cluster_field=CLUSTER_FIELD,
    method="kmeans",
    n_clusters=N_CLUSTERS,
    seed=42,
    save=True,
    verbose=False,
)

sizes = pd.Series(cluster_labels).value_counts().sort_index()
scene_by_id = dict(zip(dataset.values("id"), dataset.values("scene_id")))
summary = pd.DataFrame({
    "tiles": sizes,
    "share %": (sizes / sizes.sum() * 100).round(1),
    "scenes present": pd.Series([scene_by_id[s] for s in clustered_ids]).groupby(cluster_labels).nunique(),
})
summary.index.name = "cluster"
summary

In [ ]:
uniq_global, uniq_ids = compute_uniqueness_field(
    dataset,
    embeddings=emb,
    sample_ids=sample_ids,
    uniqueness_field=UNIQUENESS_FIELD,
    k=K_NEIGHBOURS,
    decay="exponential",
    decay_param=0.5,
    save=True,
    verbose=False,
)

In [ ]:
uniq_cluster, _ = compute_uniqueness_field_v2(
    dataset,
    embeddings=emb,
    sample_ids=sample_ids,
    uniqueness_field=UNIQUENESS_CLUSTER_FIELD,
    cluster_field=CLUSTER_FIELD,
    k=K_NEIGHBOURS,
    decay="exponential",
    decay_param=0.5,
    save=True,
    verbose=False,
)

## 8. Results

Both scores in the dataset order, the clusters on the t-SNE map, and the most unique tiles.

In [ ]:
row_of_u = {sid: i for i, sid in enumerate(uniq_ids)}
ds_ids = dataset.values("id")
g = uniq_global[[row_of_u[s] for s in ds_ids]]
c = uniq_cluster[[row_of_u[s] for s in ds_ids]]
labels = np.array([int(x) for x in dataset.values(CLUSTER_FIELD)])

print(f"global uniqueness:     mean {g.mean():.3f}, above 0.5: {(g > 0.5).sum()}")
print(f"in-cluster uniqueness: mean {c.mean():.3f}, above 0.5: {(c > 0.5).sum()}")
print(f"correlation between the two: {np.corrcoef(g, c)[0, 1]:.2f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].bar(summary.index.astype(str), summary.tiles, color="#4C72B0")
axes[0].set_title(f"Tiles per cluster (k-means, {N_CLUSTERS})")
axes[0].set_xlabel("cluster")
axes[1].hist(g, bins=40, color="#55A868")
axes[1].set_title("Global uniqueness")
axes[2].hist(c, bins=40, color="#C44E52")
axes[2].set_title("Uniqueness within cluster")
plt.tight_layout()
plt.show()

In [ ]:
points = np.asarray(dataset.load_brain_results(BRAIN_KEY).points)

fig, axes = plt.subplots(1, 2, figsize=(16, 7.5))
cmap = plt.get_cmap("tab20")
for k in sorted(set(labels)):
    m = labels == k
    axes[0].scatter(points[m, 0], points[m, 1], s=6, color=cmap(k % 20), label=str(k))
axes[0].set_title(f"t-SNE coloured by k-means cluster ({N_CLUSTERS})")
axes[0].legend(markerscale=2, fontsize=7, ncol=2, title="cluster")
sc = axes[1].scatter(points[:, 0], points[:, 1], s=6, c=g, cmap="viridis")
axes[1].set_title("t-SNE coloured by global uniqueness")
fig.colorbar(sc, ax=axes[1], label="uniqueness")
for ax in axes:
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
plt.tight_layout()
plt.show()

In [ ]:
scenes = dataset.values("scene_id")
tiles = dataset.values("tile_id")
top = np.argsort(-g)[:10]
pd.DataFrame({
    "tile_id": [tiles[i] for i in top],
    "scene_id": [scenes[i][:8] for i in top],
    "cluster": labels[top],
    "uniqueness": g[top].round(3),
    "in-cluster uniqueness": c[top].round(3),
})

The new fields are `cluster_kmeans_12`, `uniqueness_score` and `uniqueness_in_cluster_12`. In the App, filter on them, colour the t-SNE by them, and sort the grid by uniqueness to review the most unusual tiles first.

## 9. Open the FiftyOne App

Runs on your computer, so it opens in your browser.

In [ ]:
import fiftyone as fo
session = fo.launch_app(dataset)

## 8. New labels

Label new tiles in labelme and put the `.tif` + `.json` pairs in `data/labelme/images/`. Then run the cell below to rebuild the labels and the dataset. Commit and push `data/labelme/images/` so the cloud notebook gets the same labels.

In [ ]:
!uv run python export_annotations.py --index {TILES_DIR}/tiles_index.csv --labels data/labelme/images --out {TILES_DIR}/labels --tile 640
dataset = build_dataset(tiles_dir=TILES_DIR, labels_dir=f"{TILES_DIR}/labels", overwrite=True)
dataset_summary(dataset)

## 9. Later: 5 cm tiles (not run yet)

The same notebooks work for 5 cm tiles. Only the folder, the dataset name and the embedding file change:

- Tiles: `uv run python tile_cogs.py --input cog --gsd 0.05 --tile 640 --overlap 128 --output data/tiles_5cm`
- Labels: `uv run python export_annotations.py --index data/tiles_5cm/tiles_index.csv --labels data/labelme/images --out data/tiles_5cm/labels --tile 640`
- Dataset: `build_dataset(tiles_dir="data/tiles_5cm", labels_dir="data/tiles_5cm/labels", name="building_tiles_5cm", overwrite=True)`
- Embeddings (cloud): `embed.py --tiles data/tiles_5cm --out data/embeddings/5cm/dinov2_base.npz`, then `attach_embeddings(dataset, "data/embeddings/5cm/dinov2_base.npz", brain_key="dinov2_5cm")`

`attach_embeddings` refuses a file whose resolution does not match the dataset, so 5 cm embeddings can't be linked to the 10 cm dataset by mistake.